# Gaussian Splatting

## 1. Visualize Poses of Cameras (from COLMAP)

In [3]:
import numpy as np
import plotly.graph_objects as go

MODEL = "colmap_results/0_undistorted/sparse_txt"


def quat_to_rot(qw, qx, qy, qz):
    return np.array([
        [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
        [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
        [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)],
    ])


# images.txt: two lines per image; the pose is on the first.
with open(f"{MODEL}/images.txt") as f:
    lines = [l for l in f.read().splitlines() if not l.startswith("#")]

names, centers, dirs, ups = [], [], [], []
for line in lines[0::2]:
    v = line.split()
    R = quat_to_rot(*map(float, v[1:5]))      # world-to-camera rotation
    t = np.array(list(map(float, v[5:8])))
    centers.append(-R.T @ t)                   # camera center in world
    dirs.append(R.T @ [0, 0, 1])               # viewing direction in world
    ups.append(R.T @ [0, -1, 0])               # camera up (-y) in world
    names.append(v[9])
centers, dirs = np.array(centers), np.array(dirs)

# points3D.txt: X Y Z R G B ERROR, then (IMAGE_ID POINT2D_IDX) pairs.
with open(f"{MODEL}/points3D.txt") as f:
    rows = [l.split() for l in f if not l.startswith("#")]
pts = np.array([r[1:8] for r in rows], dtype=float)
track_len = np.array([(len(r) - 8) // 2 for r in rows])

center = centers.mean(axis=0)
radius = np.linalg.norm(centers - center, axis=1).max()
keep = (
    (pts[:, 6] < 1.0)                                          # low reprojection error
    & (track_len >= 3)                                         # seen in at least 3 images
    & (np.linalg.norm(pts[:, :3] - center, axis=1) < radius)   # inside the camera dome
)
pts = pts[keep]
pts = pts[np.random.choice(len(pts), min(len(pts), 20000), replace=False)]

# One line segment per camera, separated by None so they're not connected.
scale = 0.1 * np.ptp(centers, axis=0).max()
tips = centers + scale * dirs
seg = np.full((len(centers) * 3, 3), None)
seg[0::3], seg[1::3] = centers, tips

# Scene up: average of the cameras' up directions.
up = np.mean(ups, axis=0)
up /= np.linalg.norm(up)

fig = go.Figure([
    go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers",
                 marker=dict(size=1, color=[f"rgb({r:.0f},{g:.0f},{b:.0f})" for r, g, b in pts[:, 3:6]]),
                 name="points", hoverinfo="skip"),
    go.Scatter3d(x=centers[:, 0], y=centers[:, 1], z=centers[:, 2], mode="markers",
                 marker=dict(size=3, color="red"), text=names, name="cameras"),
    go.Scatter3d(x=seg[:, 0], y=seg[:, 1], z=seg[:, 2], mode="lines",
                 line=dict(color="red"), name="view directions", hoverinfo="skip"),
])
fig.update_layout(
    scene=dict(aspectmode="data"),
    scene_camera=dict(up=dict(x=up[0], y=up[1], z=up[2])),
    height=800,
)
fig.show()

## 2. Gaussian Splatting with [gsplat](https://docs.gsplat.studio/main/) library

In [ ]:
%%bash

# Using simple_trainer.
cd ~/Apps/gsplat/examples

python simple_trainer.py default \
    --data_dir ~/code/MA/gs-intro/colmap_results/0_undistorted \
    --data_factor 8 \
    --packed \
    --save_ply \
    --result_dir ~/code/MA/gs-intro/gsplat_results/tomatoes \
    --max_steps 5000 --eval_steps 2500 5000 --save_steps 2500 5000

[Parser] 166 images, taken by 166 cameras.
Downscaling images by 8x from /home/boldi/code/MA/gs-intro/colmap_results/0_undistorted/images to /home/boldi/code/MA/gs-intro/colmap_results/0_undistorted/images_8_png.


Loading EXIF exposure: 100%|██████████| 166/166 [00:00<00:00, 31045.37it/s]


[Parser] No valid EXIF exposure data found in any image.
Scene scale: 1.529745695083774
Model initialized. Number of GS: 25925
╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯


loss=0.112| sh degree=0| :  12%|█▏        | 598/5000 [00:10<01:18, 55.80it/s]

Step 600: 1332 GSs duplicated, 776 GSs split. Now having 28033 GSs.
Step 600: 20 GSs pruned. Now having 28013 GSs.


...

Training complete, disable training tab.
Viewer running... Ctrl+C to exit.


/home/boldi/miniconda3/envs/ma/lib/python3.12/site-packages/nerfview/_renderer.py:181: UserWarning: Queued a BackgroundImageMessage on a closed connection (e.g. via a handle owned by a disconnected client, or after the server stopped); it will never be delivered.
  self.client.scene.set_background_image(
Traceback (most recent call last):
  File "/home/boldi/Apps/gsplat/examples/simple_trainer.py", line 1617, in <module>
    cli(main, cfg, verbose=True)
  File "/home/boldi/Apps/gsplat/gsplat/distributed.py", line 375, in cli
    return _distributed_worker(0, 1, fn=fn, args=args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/boldi/Apps/gsplat/gsplat/distributed.py", line 310, in _distributed_worker
    fn(local_rank, world_rank, world_size, args)
  File "/home/boldi/Apps/gsplat/examples/simple_trainer.py", line 1558, in main
    time.sleep(1000000)
KeyboardInterrupt


(viser) Server stopped


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

In [6]:
%%bash

# Using simple_trainer mcmc.
cd ~/Apps/gsplat/examples

python simple_trainer.py mcmc \
    --data_dir ~/code/MA/gs-intro/colmap_results/0_undistorted \
    --data_factor 8 \
    --packed \
    --save_ply \
    --strategy.cap-max 300000 \
    --result_dir ~/code/MA/gs-intro/gsplat_results/tomatoes_mcmc \
    --max_steps 5000 --eval_steps 2500 5000 --save_steps 2500 5000

[Parser] 166 images, taken by 166 cameras.
Downscaling images by 8x from /home/boldi/code/MA/gs-intro/colmap_results/0_undistorted/images to /home/boldi/code/MA/gs-intro/colmap_results/0_undistorted/images_8_png.


Loading EXIF exposure: 100%|██████████| 166/166 [00:00<00:00, 30114.81it/s]


[Parser] No valid EXIF exposure data found in any image.
Scene scale: 1.529745695083774
Model initialized. Number of GS: 25925
╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯


loss=0.157| sh degree=0| :  12%|█▏        | 596/5000 [00:06<00:52, 84.02it/s]

Step 600: Relocated 1614 GSs.
Step 600: Added 1296 GSs. Now having 27221 GSs.


...

Training complete, disable training tab.
Viewer running... Ctrl+C to exit.


Traceback (most recent call last):
  File "/home/boldi/Apps/gsplat/examples/simple_trainer.py", line 1617, in <module>
    cli(main, cfg, verbose=True)
  File "/home/boldi/Apps/gsplat/gsplat/distributed.py", line 375, in cli
    return _distributed_worker(0, 1, fn=fn, args=args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/boldi/Apps/gsplat/gsplat/distributed.py", line 310, in _distributed_worker
    fn(local_rank, world_rank, world_size, args)
  File "/home/boldi/Apps/gsplat/examples/simple_trainer.py", line 1558, in main
    time.sleep(1000000)
KeyboardInterrupt


(viser) Server stopped


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

## gsplat Training Output

Each run writes one result folder, e.g. `gsplat_results/tomatoes/`:

| Folder | Content |
| --- | --- |
| `ckpts/` | `ckpt_<step>_rank0.pt`: the trained model, a PyTorch dict with every Gaussian parameter (means, quats, scales, opacities, SH coefficients) plus the step and config |
| `ply/` | `point_cloud_<step>.ply`: the same Gaussians in the standard 3DGS PLY format, written only with `--save_ply` |
| `renders/` | `val_step<step>_XXXX.png`: held-out views, ground truth and render side by side |
| `videos/` | `traj_<step>.mp4`: a fly-through along an interpolated camera path |
| `stats/` | `val_step<step>.json` with PSNR, SSIM, LPIPS and the Gaussian count; `train_step*.json` with timing and memory |
| `tb/` + `cfg.yml` | TensorBoard logs and the exact config the run used |

### Inspect the model interactively

```bash
cd ~/Apps/gsplat/examples
python simple_viewer.py \
    --ckpt ~/code/MA/gs-intro/gsplat_results/tomatoes/ckpts/ckpt_4999_rank0.pt \
    --port 8080
```

### Open the PLY in other tools

The PLY is the portable format. It loads in PlayCanvas's SuperSplat editor (browser-based, also crops and cleans), the Unity and Unreal plugins, Blender add-ons and web viewers. This is the file to share or embed, not the checkpoint.

### Load the parameters directly

```python
import torch

ckpt = torch.load("ckpts/ckpt_4999_rank0.pt", map_location="cpu")
splats = ckpt["splats"]
print({k: tuple(v.shape) for k, v in splats.items()})
```

The tensors can be rendered into new views with `gsplat.rasterization()`, or compared against another trainer's output.

### Read the baseline numbers

```bash
cat gsplat_results/tomatoes/stats/val_step4999.json
tensorboard --logdir gsplat_results/tomatoes/tb
```

`val_step*.json` is the reference any other trainer has to be measured against, at the same resolution and the same held-out split (every 8th image, `--test_every 8`).

In [9]:
import torch
ckpt = torch.load("gsplat_results/tomatoes_mcmc/ckpts/ckpt_4999_rank0.pt", map_location="cpu")
splats = ckpt["splats"]
print({k: tuple(v.shape) for k, v in splats.items()})

{'means': (221787, 3), 'opacities': (221787,), 'quats': (221787, 4), 'scales': (221787, 3), 'sh0': (221787, 1, 3), 'shN': (221787, 15, 3)}
